In [1]:
#import libraries
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

I0000 00:00:1783573779.635340   74203 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783573779.635804   74203 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783573779.679993   74203 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783573781.346872   74203 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [ ]:
#load data
four_df = pd.read_csv('/home/yvonne_chook/github/207-Summer26-FinalProject-MLModel/Merged_EDA/combined_data/combined_data_4h.csv')

one_df = pd.read_csv('/home/yvonne_chook/github/207-Summer26-FinalProject-MLModel/Merged_EDA/combined_data/combined_data_1h.csv')

- No delay or early departure (0)
- Less than 1 hour delay (1)
- 1–2 hour delay (2)
- 2–3 hour delay (3)
- 3–4 hour delay (4)
- 4–5 hour delay (5)
- 5–6 hour delay (6) 
- More than 6 hour delay (7)
- Flight cancelled (8)

Note to team, there should be a total of 9 outcomes

In [ ]:
#currently only 8 displayed
four_df["outcome"].unique()

In [ ]:
two_df

In [ ]:
one_df

In [ ]:
#analyze data

four_df.describe()

In [ ]:
#split train validation and test data

#Do this for 2 hr data
four_train_df = four_df[
    (four_df["date"] >= "2022-01-01") &
    (four_df["date"] <= "2024-12-31")
]

four_validation_df = four_df[
    (four_df["date"] >= "2024-01-01") &
    (four_df["date"] <= "2024-12-31")
]

four_test_df = four_df[
    (four_df["date"] >= "2025-01-01") &
    (four_df["date"] <= "2026-06-09")
]

#Do this for 4 hr data
one_train_df = one_df[
    (one_df["date"] >= "2022-01-01") &
    (one_df["date"] <= "2024-12-31")
]

one_validation_df = one_df[
    (one_df["date"] >= "2024-01-01") &
    (one_df["date"] <= "2024-12-31")
]

one_test_df = one_df[
    (one_df["date"] >= "2025-01-01") &
    (one_df["date"] <= "2026-06-09")
]

#print shape of each
print("2hr data:")
print("Train shape:", four_train_df.shape)
print("Validation shape:", four_validation_df.shape)
print("Test shape:", four_test_df.shape)

print("\n1hr data:")
print("Train shape:", one_train_df.shape)
print("Validation shape:", one_validation_df.shape)
print("Test shape:", one_test_df.shape)

In [ ]:
four_train_df.columns

In [ ]:
#feature selection

features = [
    "scheduled_elapsed_time_minutes",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sched_dep_hour",
    "days_until_holiday",
    "days_from_holiday",
    "year_mfr",
    "no_seats",
    "visibility",
    "ceiling_height",
    "wind_speed",
    "wind_direction",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "altimeter",
    "aircraft_age",
    "week_num",
    "temperature_dewpoint_spread",
    "airport_delay_average_1h",
    "airport_delay_stddev_1h",
    "airport_departures_observed_1h"
]

target = "departure_delay_minutes"


**2 hour linear regression baseline**

In [ ]:
# create X and y
four_X_train = four_train_df[features]
four_y_train = four_train_df[target]

four_X_val = four_validation_df[features]
four_y_val = four_validation_df[target]

four_X_test = four_test_df[features]
four_y_test = four_test_df[target]

# scale features
scaler = StandardScaler()

four_X_train_scaled = scaler.fit_transform(four_X_train)
four_X_val_scaled = scaler.transform(four_X_val)
four_X_test_scaled = scaler.transform(four_X_test)

In [ ]:
#build model
def build_model(num_features, learning_rate):
    tf.keras.backend.clear_session()
    tf.random.set_seed(0)

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(num_features,)),
        tf.keras.layers.Dense(units=1, use_bias=True)
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# train model

four_model = build_model(
    num_features=four_X_train_scaled.shape[1],
    learning_rate=0.001
)

four_trained_model = four_model.fit(
    four_X_train_scaled,
    four_y_train,
    validation_data=(four_X_val_scaled, four_y_val),
    epochs=100,
    batch_size=32,
    verbose = 0
)

#plot output
#plot
plt.plot(four_trained_model.history['loss'], label="Training Loss")
plt.plot(four_trained_model.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Values Over Epochs")
plt.legend()
plt.show()


In [ ]:
#Obtain weight and bias
four_weights, four_bias = four_model.layers[0].get_weights()

#calulate train and test mse

four_train_mse, four_train_mae = four_model.evaluate(
    four_X_train_scaled,
    four_y_train,
    verbose=0
)

# Calculate test MSE and MAE
four_test_mse, four_test_mae = four_model.evaluate(
    four_X_test_scaled,
    four_y_test,
    verbose=0
)

# Learned parameters
print(f"Learned Weights:\n{four_weights}")
print(f"\nLearned Bias:\n{four_bias}")

print(f"Train MSE: {four_train_mse:.3f}")
print(f"Train MAE: {four_train_mae:.3f}")

print(f"Test MSE: {four_test_mse:.3f}")
print(f"Test MAE: {four_test_mae:.3f}")

print(f"Difference between test and train MSE: {abs(four_train_mse - four_test_mse):.3f}")

**1 hour linear regression**

In [ ]:
# create X and y
one_X_train = one_train_df[features]
one_y_train = one_train_df[target]

one_X_val = one_validation_df[features]
one_y_val = one_validation_df[target]

one_X_test = one_test_df[features]
one_y_test = one_test_df[target]

# scale features
scaler = StandardScaler()

one_X_train_scaled = scaler.fit_transform(one_X_train)
one_X_val_scaled = scaler.transform(one_X_val)
one_X_test_scaled = scaler.transform(one_X_test)

In [ ]:
# train model

one_model = build_model(
    num_features=one_X_train_scaled.shape[1],
    learning_rate=0.01
)

one_trained_model = one_model.fit(
    one_X_train_scaled,
    one_y_train,
    validation_data=(one_X_val_scaled, one_y_val),
    epochs=50,
    batch_size=32,
    verbose = 0
)

#plot output
#plot
plt.plot(one_trained_model.history['loss'], label="Training Loss")
plt.plot(one_trained_model.history['val_loss'], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Values Over Epochs")
plt.legend()
plt.show()

In [ ]:
#Obtain weight and bias
one_weights, one_bias = one_model.layers[0].get_weights()

#calulate train and test mse

one_train_mse, one_train_mae = one_model.evaluate(
    one_X_train_scaled,
    one_y_train,
    verbose=0
)

# Calculate test MSE and MAE
one_test_mse, one_test_mae = one_model.evaluate(
    one_X_test_scaled,
    one_y_test,
    verbose=0
)

# Learned parameters
print(f"Learned Weights:\n{one_weights}")
print(f"\nLearned Bias:\n{one_bias}")

print(f"Train MSE: {one_train_mse:.3f}")
print(f"Train MAE: {one_train_mae:.3f}")

print(f"Test MSE: {one_test_mse:.3f}")
print(f"Test MAE: {one_test_mae:.3f}")

print(f"Difference between test and train MSE: {abs(one_train_mse - one_test_mse):.3f}")

**LOGISTIC REGRESSION**

In [ ]:
#define features
target = "outcome"

numeric_features = [
    "scheduled_elapsed_time_minutes",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sched_dep_hour",
    "days_until_holiday",
    "days_from_holiday",
    "year_mfr",
    "no_seats",
    "visibility",
    "ceiling_height",
    "wind_speed",
    "wind_direction",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "altimeter",
    "aircraft_age",
    "week_num",
    "temperature_dewpoint_spread",
    "airport_delay_average_1h",
    "airport_delay_stddev_1h",
    "airport_departures_observed_1h"
]

categorical_features = [
    "carrier_code",
    "destination_airport",
    "model"
]

**4hr Logistic Regression**

In [ ]:
four_X_train = four_train_df[numeric_features + categorical_features]
four_y_train = four_train_df[target]

four_X_val = four_validation_df[numeric_features + categorical_features]
four_y_val = four_validation_df[target]

four_X_test = four_test_df[numeric_features + categorical_features]
four_y_test = four_test_df[target]

In [ ]:
#build preprocessing pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

four_X_train_processed = preprocessor.fit_transform(four_X_train)

four_X_val_processed = preprocessor.transform(four_X_val)

four_X_test_processed = preprocessor.transform(four_X_test)

#check sahpe

print("Train shape:", four_X_train_processed.shape)
print("Validation shape:", four_X_val_processed.shape)
print("Test shape:", four_X_test_processed.shape)

In [ ]:
#build model

def build_model(num_features, learning_rate):
    """Return a simple multiclass logistic regression model using Keras."""

    tf.keras.backend.clear_session()
    tf.random.set_seed(0)

    model = tf.keras.Sequential()

    model.add(tf.keras.Input(shape=(num_features,), name="Input"))

    model.add(tf.keras.layers.Dense(
        units=9,
        use_bias=True,
        activation="softmax",
        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01),
        bias_initializer=tf.keras.initializers.RandomNormal(stddev=0.01),
        name="Output"
    ))

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        metrics=["accuracy"]
    )

    return model

In [ ]:
#train

four_model = build_model(
    num_features=four_X_train_processed.shape[1],
    learning_rate=0.01
)

four_trained_model = four_model.fit(
    four_X_train_processed,
    four_y_train,
    validation_data=(four_X_val_processed, four_y_val),
    epochs=50,
    batch_size=32,
    verbose=0
)

In [ ]:
#plot

# Get number of epochs actually trained
num_epochs = len(four_trained_model.history["loss"])
epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Subplot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, four_trained_model.history["loss"], label="Training Loss")
plt.plot(epochs, four_trained_model.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()

# Subplot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, four_trained_model.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, four_trained_model.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
four_train_loss, four_train_acc = four_model.evaluate(
    four_X_train_processed,
    four_y_train,
    verbose=0
)

four_val_loss, four_val_acc = four_model.evaluate(
    four_X_val_processed,
    four_y_val,
    verbose=0
)

four_test_loss, four_test_acc = four_model.evaluate(
    four_X_test_processed,
    four_y_test,
    verbose=0
)

print(f"Train Loss: {four_train_loss:.3f}")
print(f"Train Accuracy: {four_train_acc:.3f}")

print(f"\nValidation Loss: {four_val_loss:.3f}")
print(f"Validation Accuracy: {four_val_acc:.3f}")

print(f"\nTest Loss: {four_test_loss:.3f}")
print(f"Test Accuracy: {four_test_acc:.3f}")